# Section 1: project outline
Russia 2018 World Cup — Economic Impact Analysis
Treatment Group: Russia
Control Group: Poland
#Years: 2013–2023

DATA PERIOD
We will gather 5 years prior and after the world cup to analyze which will depend for each host
Germany - 2006
South Africa -  2010
Brazil - 2014 
Russia - 2018

Method: 5 Years Before vs 5 Years After (excluding 2018)
This notebook analyzes the economic impact of the 2018 FIFA World Cup on Russia using Poland as a control group.
We compare:

GDP (USD)
GDP Growth (%)
Inflation (%)
Unemployment (%)
Government Debt (% of GDP)
Tourism Revenue (USD)
Tourism Revenue as % of GDP
Tourist Arrivals
Hotel Occupancy (if available)

Data Sources:
1. World Bank API
2. IMF API (government debt)
3. Web scraping (stadium data, hotel occupancy)
4. The Moscow Time
5. wikipedia
6. UN Tourism csv

SECTION 2: DATA PLAN AND INDICATOR CODES
World Bank indicators: https://data.worldbank.org/indicator?tab=all

Example with Germany:
Germany country code in World Bank API: DEU
GDP current US$ - NY.GDP.MKTP.CD
GDP growth annual (%) - NY.GDP.MKTP.KD.ZG
Inflation rate (%) - FP.CPI.TOTL.ZG
Unemployment rate (%) - SL.UEM.TOTL.ZS
International Tourism arrivals - ST.INT.ARVL
Tourism receipts current US$ - ST.INT.RCPT.CD
Tourism receipts as % of GDP - calculated as tourism receipts / GDP * 100

In [1]:
# # SECTION 3: IMPORT LIBRARIES AND DISPLAY SETTINGS
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List
import sys
import os
import re
from IPython.display import Image
# Display large numbers with commas and two decimal places
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
# SECTION 3: WORLD BANK API HELPER FUNCTION
# Retrieve one indicator from the World Bank API and prepare it for analysis
def get_worldbank_indicator(country, indicator, metric_name):

    # Build the API request using the country and indicator codes
    url_worldbank = f"https://api.worldbank.org/v2/country/{country}/indicator/{indicator}?format=json&per_page=500"
  
    # Request the data and stop if the API returns an error
    response = requests.get(url_worldbank)
    response.raise_for_status()

    # Convert the API response into a two-column DataFrame
    data = response.json()

    df_country = pd.DataFrame(data[1])

    df_country = df_country[["date", "value"]]

    df_country = df_country.rename(columns={
        "date": "year",
        "value": metric_name
    })

    # Convert year to an integer and keep the 2013 - 2023 study period
    df_country["year"] = df_country["year"].astype(int)

    df_country = df_country[df_country["year"].between(2013, 2023)]

    return df_country.sort_values("year").reset_index(drop=True)

In [3]:
# SECTION 4: IMF API HELPER FUNCTION
# Retrieve one indicator from the IMF DataMapper API
def get_IMF_indicator(country, indicator, metric_name):

    # Build and send the request for the selected IMF indicator
    url_IMF = f"https://www.imf.org/external/datamapper/api/v2/{indicator}"

    response = requests.get(url_IMF)
    response.raise_for_status()

    data = response.json()

    # Select the year-value dictionary for the requested country
    country_data = data["values"][indicator][country]

    df_IMF_country = pd.DataFrame(
        country_data.items(),
        columns=["year", metric_name]
    )

    # Convert year to an integer and keep the 2013-2023 study period
    df_IMF_country["year"] = df_IMF_country["year"].astype(int)

    df_IMF_country = df_IMF_country[
        df_IMF_country["year"].between(2013, 2023)
    ]

    return df_IMF_country.sort_values("year").reset_index(drop=True)

In [4]:
# SECTION 5: BUILD EACH COUNTRY'S DATASET
# Build one complete macroeconomic and tourism dataset for a country
def build_country_dataset(wb_code, imf_code, country_name):

    # Actual GDP measured in current US dollars
    gdp = get_worldbank_indicator(
        wb_code,
        "NY.GDP.MKTP.CD",
        "gdp_current_usd"
    )

    # Annual GDP growth rate expressed as a percentage
    gdp_growth = get_worldbank_indicator(
        wb_code,
        "NY.GDP.MKTP.KD.ZG",
        "gdp_growth_pct"
    )

    # Annual inflation rate expressed as a percentage
    inflation = get_worldbank_indicator(
        wb_code,
        "FP.CPI.TOTL.ZG",
        "inflation_pct"
    )

    # Unemployment as a percentage of the labour force
    unemployment = get_worldbank_indicator(
        wb_code,
        "SL.UEM.TOTL.ZS",
        "unemployment_pct"
    )

    # International tourism revenue measured in current US dollars
    tourism_receipts = get_worldbank_indicator(
        wb_code,
        "ST.INT.RCPT.CD",
        "tourism_revenue_current_usd"
    )

    # tourism arrival
    tourism_arrival = get_worldbank_indicator(
        wb_code,
        "ST.INT.ARVL",
        "tourism_arrivals"
    )

    # Government debt expressed as a percentage of GDP
    government_debt = get_IMF_indicator(
        imf_code,
        "GG_DEBT_GDP",
        "government_debt_pct_gdp"
    )

    # Merge all indicators by year; outer joins preserve incomplete years
    country_df = (
        gdp
        .merge(gdp_growth, on="year", how="outer")
        .merge(inflation, on="year", how="outer")
        .merge(unemployment, on="year", how="outer")
        .merge(tourism_receipts, on="year", how="outer")
        .merge(government_debt, on="year", how="outer")
        .merge(tourism_arrival, on="year", how="outer")
    )

    # Calculate how much tourism revenue contributes relative to GDP
    country_df["tourism_revenue_pct_gdp"] = (
        country_df["tourism_revenue_current_usd"]
        / country_df["gdp_current_usd"]
        * 100
    )

    # Place the country name immediately after the year column
    country_df.insert(1, "country", country_name)

    return country_df

In [5]:
#  SECTION 6: USING RUSSIA AS HOST AND POLAND AS CONTROL

# COMBINE RUSSIA AND POLAND
# Build the individual country datasets using their API country codes
russia_df = build_country_dataset("RUS", "RUS", "Russia")
poland_df = build_country_dataset("POL", "POL", "Poland")

# Stack both countries into one analysis-ready table
russia_poland_df = pd.concat(
    [russia_df, poland_df],
    ignore_index=True
)

# Organize the rows by country and year
russia_poland_df = russia_poland_df.sort_values(
    ["country", "year"]
).reset_index(drop=True)

russia_poland_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_arrivals,tourism_revenue_pct_gdp
0,2013,Poland,"518,179,836,405.36",0.68,0.99,10.29,"12,242,000,000.00",56.86,"72,310,000.00",2.36
1,2014,Poland,"542,134,167,178.63",3.92,0.05,8.97,"12,691,000,000.00",51.13,"73,750,000.00",2.34
2,2015,Poland,"480,054,118,583.37",4.43,-0.87,7.47,"11,164,000,000.00",51.06,"77,743,000.00",2.33
3,2016,Poland,"473,259,583,969.64",3.03,-0.66,6.14,"11,922,000,000.00",54.13,"80,476,000.00",2.52
4,2017,Poland,"528,356,676,667.21",5.15,2.08,4.87,"13,925,000,000.00",50.44,"83,804,000.00",2.64
5,2018,Poland,"594,616,687,350.02",6.25,1.81,3.83,"15,569,000,000.00",48.23,"85,946,000.00",2.62
6,2019,Poland,"602,683,770,144.88",4.58,2.23,3.27,"15,712,000,000.00",45.21,"88,515,000.00",2.61
7,2020,Poland,"605,914,237,903.74",-2.04,3.37,3.15,"8,379,000,000.00",56.58,NaN,1.38
8,2021,Poland,"689,170,230,665.35",6.93,5.06,3.27,NaN,53.01,NaN,NaN
9,2022,Poland,"695,607,470,875.28",5.26,14.43,2.81,NaN,48.79,NaN,NaN


In [6]:
# SECTION 7: Introducing UN tourist arrival data to fill in the NaN values
# Not all the Tourist arrival and Tourism revenue info was unavialble for both Russia and poland since there was a change in data collection and measures

# SECTION 8: CLEAN INTERNATIONAL TOURIST-ARRIVAL DATA
# Load international tourist-arrival data from Our World in Data
url_UN = "https://ourworldindata.org/grapher/international-tourist-trips.csv?v=1&csvType=full&useColumnShortNames=false"


df_UN_tourism_arrivals = pd.read_csv("international-tourist-trips.csv")

# Keep Russia, Poland, and the 2013-2023 study period
tourism_arrivals_fixed = df_UN_tourism_arrivals[
    (df_UN_tourism_arrivals["Entity"].isin(["Russia", "Poland"])) &
    (df_UN_tourism_arrivals["Year"].between(2013, 2023))
].copy()

# Select only the fields required for the final dataset
tourism_arrivals_fixed = tourism_arrivals_fixed[[
    "Entity",
    "Year",
    "Arrivals of tourists from abroad"
]]

# Standardize column names so they match the main dataset
tourism_arrivals_fixed.columns = [
    "country",
    "year",
    "tourism_arrivals"
]

# Organize the rows and create a clean sequential index
tourism_arrivals_fixed = tourism_arrivals_fixed.sort_values(
    ["country", "year"]
).reset_index(drop=True)

tourism_arrivals_fixed

,country,year,tourism_arrivals
0,Poland,2013,"15,800,000.00"
1,Poland,2014,"16,000,000.00"
2,Poland,2015,"16,728,000.00"
3,Poland,2016,"17,471,000.00"
4,Poland,2017,"18,258,000.00"
5,Poland,2018,"19,622,000.00"
6,Poland,2019,"21,158,000.00"
7,Poland,2020,"8,418,000.00"
8,Poland,2021,"9,722,000.00"
9,Poland,2022,"15,948,000.00"


In [7]:
# Step 8: we Mapped the actual 2021-2023 Poland arrival data from your CSV into the main dataframe
# We set 'year' and 'country' as the index on both to align them perfectly
main_df_indexed = russia_poland_df.set_index(['year', 'country'])
csv_df_indexed = tourism_arrivals_fixed.rename(columns={'tourism_arrivals': 'arrivals_from_csv'}).set_index(['year', 'country'])

# we Update the tourism_arrivals column where the CSV has data
main_df_indexed['tourism_arrivals'] = main_df_indexed['tourism_arrivals'].fillna(csv_df_indexed['arrivals_from_csv'])

# Reset the index back to normal
russia_poland_df = main_df_indexed.reset_index()

# Step 2: Now run the groupby forward-fill to clean up Russia's remaining gaps and the revenue column
russia_poland_df['tourism_revenue_current_usd'] = russia_poland_df.groupby('country')['tourism_revenue_current_usd'].ffill()
russia_poland_df['tourism_arrivals'] = russia_poland_df.groupby('country')['tourism_arrivals'].ffill()

# View your complete dataframe
russia_poland_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_arrivals,tourism_revenue_pct_gdp
0,2013,Poland,"518,179,836,405.36",0.68,0.99,10.29,"12,242,000,000.00",56.86,"72,310,000.00",2.36
1,2014,Poland,"542,134,167,178.63",3.92,0.05,8.97,"12,691,000,000.00",51.13,"73,750,000.00",2.34
2,2015,Poland,"480,054,118,583.37",4.43,-0.87,7.47,"11,164,000,000.00",51.06,"77,743,000.00",2.33
3,2016,Poland,"473,259,583,969.64",3.03,-0.66,6.14,"11,922,000,000.00",54.13,"80,476,000.00",2.52
4,2017,Poland,"528,356,676,667.21",5.15,2.08,4.87,"13,925,000,000.00",50.44,"83,804,000.00",2.64
5,2018,Poland,"594,616,687,350.02",6.25,1.81,3.83,"15,569,000,000.00",48.23,"85,946,000.00",2.62
6,2019,Poland,"602,683,770,144.88",4.58,2.23,3.27,"15,712,000,000.00",45.21,"88,515,000.00",2.61
7,2020,Poland,"605,914,237,903.74",-2.04,3.37,3.15,"8,379,000,000.00",56.58,"8,418,000.00",1.38
8,2021,Poland,"689,170,230,665.35",6.93,5.06,3.27,"8,379,000,000.00",53.01,"9,722,000.00",NaN
9,2022,Poland,"695,607,470,875.28",5.26,14.43,2.81,"8,379,000,000.00",48.79,"15,948,000.00",NaN


In [8]:
# SECTION 9: continue to fill the NaN Values
# After filling the Nan values for both Poland and Russia tourism revenue as a percentage of GDP couldnt update itself
# We reclaute the tourism revenue as a percentage of GDP
russia_poland_df['tourism_revenue_pct_gdp'] = (russia_poland_df['tourism_revenue_current_usd'] / russia_poland_df['gdp_current_usd']) * 100

# View the updated dataframe
russia_poland_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_arrivals,tourism_revenue_pct_gdp
0,2013,Poland,"518,179,836,405.36",0.68,0.99,10.29,"12,242,000,000.00",56.86,"72,310,000.00",2.36
1,2014,Poland,"542,134,167,178.63",3.92,0.05,8.97,"12,691,000,000.00",51.13,"73,750,000.00",2.34
2,2015,Poland,"480,054,118,583.37",4.43,-0.87,7.47,"11,164,000,000.00",51.06,"77,743,000.00",2.33
3,2016,Poland,"473,259,583,969.64",3.03,-0.66,6.14,"11,922,000,000.00",54.13,"80,476,000.00",2.52
4,2017,Poland,"528,356,676,667.21",5.15,2.08,4.87,"13,925,000,000.00",50.44,"83,804,000.00",2.64
5,2018,Poland,"594,616,687,350.02",6.25,1.81,3.83,"15,569,000,000.00",48.23,"85,946,000.00",2.62
6,2019,Poland,"602,683,770,144.88",4.58,2.23,3.27,"15,712,000,000.00",45.21,"88,515,000.00",2.61
7,2020,Poland,"605,914,237,903.74",-2.04,3.37,3.15,"8,379,000,000.00",56.58,"8,418,000.00",1.38
8,2021,Poland,"689,170,230,665.35",6.93,5.06,3.27,"8,379,000,000.00",53.01,"9,722,000.00",1.22
9,2022,Poland,"695,607,470,875.28",5.26,14.43,2.81,"8,379,000,000.00",48.79,"15,948,000.00",1.20


In [9]:
# SECTION 10: CREATE BEFORE/AFTER PERIODS FROM THE YEAR COLUMN

# Years before the 2018 World Cup are labeled "Before"
# Years from 2018 onward are labeled "After"
russia_poland_df["period"] = np.where(
    russia_poland_df["year"] < 2018,
    "Before",
    "After"
)

russia_poland_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_arrivals,tourism_revenue_pct_gdp,period
0,2013,Poland,"518,179,836,405.36",0.68,0.99,10.29,"12,242,000,000.00",56.86,"72,310,000.00",2.36,Before
1,2014,Poland,"542,134,167,178.63",3.92,0.05,8.97,"12,691,000,000.00",51.13,"73,750,000.00",2.34,Before
2,2015,Poland,"480,054,118,583.37",4.43,-0.87,7.47,"11,164,000,000.00",51.06,"77,743,000.00",2.33,Before
3,2016,Poland,"473,259,583,969.64",3.03,-0.66,6.14,"11,922,000,000.00",54.13,"80,476,000.00",2.52,Before
4,2017,Poland,"528,356,676,667.21",5.15,2.08,4.87,"13,925,000,000.00",50.44,"83,804,000.00",2.64,Before
5,2018,Poland,"594,616,687,350.02",6.25,1.81,3.83,"15,569,000,000.00",48.23,"85,946,000.00",2.62,After
6,2019,Poland,"602,683,770,144.88",4.58,2.23,3.27,"15,712,000,000.00",45.21,"88,515,000.00",2.61,After
7,2020,Poland,"605,914,237,903.74",-2.04,3.37,3.15,"8,379,000,000.00",56.58,"8,418,000.00",1.38,After
8,2021,Poland,"689,170,230,665.35",6.93,5.06,3.27,"8,379,000,000.00",53.01,"9,722,000.00",1.22,After
9,2022,Poland,"695,607,470,875.28",5.26,14.43,2.81,"8,379,000,000.00",48.79,"15,948,000.00",1.20,After


In [10]:
# SECTION 11: CALCULATE AVERAGE INDICATORS BY COUNTRY AND PERIOD
# Select the indicators used to compare the Before and After periods
metrics = [
    "gdp_current_usd",
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_pct",
    "government_debt_pct_gdp",
    "tourism_revenue_current_usd",
    "tourism_arrivals"
]

# Calculate the mean of each indicator for every country-period combination
avg_table = russia_poland_df.pivot_table(
    index="country",
    columns="period",
    values=metrics,
    aggfunc="mean"
)

# Round the displayed results while keeping the underlying values unchanged
avg_table.round(2)

gdp_current_usd                      gdp_growth_pct         \
period                 After               Before          After Before   
country                                                                   
Poland    666,740,598,389.23   508,396,876,560.84           3.54   3.44   
Russia  1,839,304,039,412.02 1,713,235,912,281.82           1.81   0.51   

        government_debt_pct_gdp        inflation_pct        tourism_arrivals  \
period                    After Before         After Before            After   
country                                                                        
Poland                    50.26  52.72          6.40   0.32    37,922,666.67   
Russia                    16.84  14.39          6.17   8.17    12,401,000.00   

                      tourism_revenue_current_usd                    \
period         Before                       After            Before   
country                                                               
Poland  77,616,600.00           10,799,500,000.00 12,388,800,000.00   
Russia  29,180,600.00            9,302,333,333.33 16,119,000,000.00   

        unemployment_pct         
period             After Before  
country                          
Poland              3.18   7.55  
Russia              4.45   5.43

In [11]:
# SECTION 12: MEASURE THE CHANGE FROM BEFORE TO AFTER
# Extract the country averages for each period from the pivot table

#Cross sections (xs) works this way:
    #   dataframe.xs(
    #       label_to_find,
    #       level=where_to_find_it, --> in this case
    #        axis=rows_or_columns
    #   )

before = avg_table.xs(
    "Before",
    level=1,
    axis=1
)

after = avg_table.xs(
    "After",
    level=1,
    axis=1
)

# Positive values indicate an increase; negative values indicate a decrease
change_table = after - before

change_table.round(2)

,gdp_current_usd,gdp_growth_pct,government_debt_pct_gdp,inflation_pct,tourism_arrivals,tourism_revenue_current_usd,unemployment_pct
country,,,,,,,
Poland,"158,343,721,828.39",0.09,-2.47,6.09,"-39,693,933.33","-1,589,300,000.00",-4.37
Russia,"126,068,127,130.20",1.30,2.45,-1.99,"-16,779,600.00","-6,816,666,666.67",-0.98


In [12]:
# SECTION 13: ESTIMATE RUSSIA'S RELATIVE WORLD CUP IMPACT
# We did a Difference-in-Differences (DiD) analysis because both country went through COVID-19 and Russia with the WAR
# Subtract Poland's change from Russia's change to create a comparison group
impact_table = (
    change_table.loc["Russia"]
    -
    change_table.loc["Poland"]
)

# Convert the resulting Series into a clearly labelled one-column table
impact_table = impact_table.to_frame(
    name="Russia_minus_Poland"
)

impact_table.round(2)

,Russia_minus_Poland
gdp_current_usd,"-32,275,594,698.19"
gdp_growth_pct,1.21
government_debt_pct_gdp,4.92
inflation_pct,-8.08
tourism_arrivals,"22,914,333.33"
tourism_revenue_current_usd,"-5,227,366,666.67"
unemployment_pct,3.39



# SECTION 14: EXTRA CODE FOR RUSSIAN WORLD CUP STUDY

Russia 2018 World Cup - Estimated Total Investment ≈ $11.0 Billion to $13.2 Billion (2018)

The Russian government estimates total spending on the 2018 FIFA World Cup at roughly 678–683 billion rubles ($10.7 to $13.2 billion).

Russia invested approximately $3.5 billion to $4.22 billion (200–265 billion rubles) explicitly in stadium construction and modernization for the 12 tournament venues.

Additional transport and support infrastructure investments (including rail, airports, roads, and utilities) were estimated at roughly $6.0 billion (353 billion rubles) according to official allocations.

# Pivots to get
#       GDP 
#       Tourism Revenue
#       Tourist Arrivals
#       % GDP growth
#       % Government debt as part of GDP
#       % Tourism as part of GDP


In [13]:
#SECTION 15: Create a GDP table with years as rows and countries as columns
# This makes it easy to compare Russia and Poland GDP year by year
gdp_table = russia_poland_df.pivot_table(
    index="year",
    columns="country",
    values="gdp_current_usd",
    aggfunc="mean"
)

gdp_table

country,Poland,Russia
year,,
2013,"518,179,836,405.36","2,292,470,078,346.22"
2014,"542,134,167,178.63","2,059,241,589,895.01"
2015,"480,054,118,583.37","1,363,482,182,197.71"
2016,"473,259,583,969.64","1,276,786,350,881.14"
2017,"528,356,676,667.21","1,574,199,360,089.00"
2018,"594,616,687,350.02","1,657,328,773,461.31"
2019,"602,683,770,144.88","1,693,115,002,708.32"
2020,"605,914,237,903.74","1,493,075,894,362.14"
2021,"689,170,230,665.35","1,829,186,719,575.10"


In [14]:
# SECTION 16: Create a tourism revenue table with years as rows and countries as columns
# This shows how much tourism revenue each country earned each year
tourism_revenue_table = russia_poland_df.pivot_table(
    index="year",
    columns="country",
    values="tourism_revenue_current_usd",
    aggfunc="mean"
)

tourism_revenue_table

country,Poland,Russia
year,,
2013,"12,242,000,000.00","20,198,000,000.00"
2014,"12,691,000,000.00","19,451,000,000.00"
2015,"11,164,000,000.00","13,186,000,000.00"
2016,"11,922,000,000.00","12,822,000,000.00"
2017,"13,925,000,000.00","14,938,000,000.00"
2018,"15,569,000,000.00","18,735,000,000.00"
2019,"15,712,000,000.00","17,235,000,000.00"
2020,"8,379,000,000.00","4,961,000,000.00"
2021,"8,379,000,000.00","4,961,000,000.00"


In [15]:
# SECTION 17: Create a tourist arrivals table with years as rows and countries as columns
# This compares the number of international tourist arrivals each year
tourist_arrivals_table = russia_poland_df.pivot_table(
    index="year",
    columns="country",
    values="tourism_arrivals",
    aggfunc="mean"
)

tourist_arrivals_table

country,Poland,Russia
year,,
2013,"72,310,000.00","30,792,000.00"
2014,"73,750,000.00","32,421,000.00"
2015,"77,743,000.00","33,729,000.00"
2016,"80,476,000.00","24,571,000.00"
2017,"83,804,000.00","24,390,000.00"
2018,"85,946,000.00","24,551,000.00"
2019,"88,515,000.00","24,419,000.00"
2020,"8,418,000.00","6,359,000.00"
2021,"9,722,000.00","6,359,000.00"


In [16]:
# SECTION 18: Create a GDP growth table with years as rows and countries as columns
# This compares each country's yearly GDP growth percentage
gdp_growth_table = russia_poland_df.pivot_table(
    index="year",
    columns="country",
    values="gdp_growth_pct",
    aggfunc="mean"
)

gdp_growth_table

country,Poland,Russia
year,,
2013,0.68,1.76
2014,3.92,0.74
2015,4.43,-1.97
2016,3.03,0.19
2017,5.15,1.83
2018,6.25,2.81
2019,4.58,2.20
2020,-2.04,-2.65
2021,6.93,5.87


In [17]:
# SECTION 19: Create a government debt table with years as rows and countries as columns
# This compares government debt as a percentage of GDP for each country
gvt_debt= russia_poland_df.pivot_table(
    index="year",
    columns= ["country"],
    values="government_debt_pct_gdp",
)

gvt_debt

country,Poland,Russia
year,,
2013,56.86,12.35
2014,51.13,15.14
2015,51.06,15.29
2016,54.13,14.85
2017,50.44,14.31
2018,48.23,13.62
2019,45.21,13.75
2020,56.58,19.16
2021,53.01,16.52


In [18]:
# SECTION 20: Create a tourism revenue percentage-of-GDP table by year and country
# This shows how important tourism revenue was relative to each country's GDP
tourism_pct_gdp = russia_poland_df.pivot_table(
    index="year",
    columns= ["country"],
    values="tourism_revenue_pct_gdp",
)

tourism_pct_gdp

country,Poland,Russia
year,,
2013,2.36,0.88
2014,2.34,0.94
2015,2.33,0.97
2016,2.52,1.00
2017,2.64,0.95
2018,2.62,1.13
2019,2.61,1.02
2020,1.38,0.33
2021,1.22,0.27


In [19]:
# SECTION 21: Function to find the percentage change between 2 years in a dataframe.

def calculate_percent_change(df, start_year, end_year, value_column):
    start_value = df.loc[df["year"] == start_year, value_column].iloc[0]
    end_value = df.loc[df["year"] == end_year, value_column].iloc[0]

    return ((end_value - start_value) / start_value) * 100

In [20]:
# SECTION 22: Calculate the percentage change in GDP during the period BEFORE the World Cup (2013-2017)
calculate_percent_change(russia_poland_df,2013, 2017,"gdp_current_usd" )

np.float64(1.9639591406042107)

In [21]:
# SECTION 23: Calculate the percentage change in GDP during the period AFTER the World Cup (2013-2017)
calculate_percent_change(russia_poland_df,2019, 2023,"gdp_current_usd" )

np.float64(34.80555369872274)

In [22]:
# SECTION 24: Scrape the Russia 2018 World Cup Logo
url = "https://en.wikipedia.org/wiki/2018_FIFA_World_Cup"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "lxml")

# Find ANY infobox
infobox = soup.find("table", class_=lambda x: x and "infobox" in x)

if infobox:
    logo_img = infobox.find("img")
    if logo_img:
        logo_url = "https:" + logo_img["src"]
        print("Logo URL:", logo_url)
    else:
        print("Image not found inside infobox.")
else:
    print("Infobox not found.")

from IPython.display import Image
Image(url=logo_url)

Logo URL: https://upload.wikimedia.org/wikipedia/en/thumb/6/67/2018_FIFA_World_Cup.svg/250px-2018_FIFA_World_Cup.svg.png


In [23]:
# SECTION 25: Scrape the winning country
url = "https://en.wikipedia.org/wiki/2018_FIFA_World_Cup"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "lxml")

infobox = soup.find("table", class_=lambda x: x and "infobox" in x)

winner = None
for row in infobox.find_all("tr"):
    header = row.find("th")
    if header and "Champions" in header.text:
        winner = row.find("td").text.strip()
        break

print("Winner:", winner)

Winner: France (2nd title)


In [24]:
# SECTION 25: Scrape the  stadium

# URL for the specific 2018 World Cup Final page
final_url = "https://en.wikipedia.org/wiki/2018_FIFA_World_Cup_Final"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

response = requests.get(final_url, headers=headers)
soup = BeautifulSoup(response.text, "lxml")

# Find the specific image of France celebrating with the trophy
# We look for the image containing 'France_celebrating' or the main gallery photo
celebration_img = soup.find("img", alt=lambda x: x and "celebrat" in x.lower())

if not celebration_img:
    # Backup selector: find an image with 'trophy' or 'winner' in the filename/alt text
    celebration_img = soup.find("img", src=lambda x: x and "France" in x and "World_Cup" in x)

if celebration_img:
    # Build the full image URL
    winner_img_url = "https:" + celebration_img["src"]
    print(f"Found celebration image URL: {winner_img_url}")
    
    # Display the image with a custom width so it fits cleanly in your notebook
    display(Image(url=winner_img_url, width=600))
else:
    # Absolute direct link backup if Wikipedia structure changes slightly
    backup_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a9/France_celebrating_2018_World_Cup.jpg/640px-France_celebrating_2018_World_Cup.jpg"
    print("Using direct backup URL...")
    display(Image(url=backup_url, width=600))


Found celebration image URL: https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/2018_World_Cup_Final_-_France_v_Croatia_-_1st_Half.jpg/250px-2018_World_Cup_Final_-_France_v_Croatia_-_1st_Half.jpg
